In [8]:
import pandas as pd
df = pd.read_csv("retrievable_peptides.tsv", sep="\t")
df.head()

,precursor_protein,evidence_peptides_scans
0,P25021,"[('APNGTASSFCLDSTACK', 5159, 'non_standard_mod..."
1,Q96DZ7,"[('VALKDGPFCMF', 20487, 'HLA1', True, 'Missed'..."
2,P43116,"[('GNASNDSQSEDCETR', 5433, 'non_standard_modif..."
3,O95838,"[('KLQPSLNSGR', 2991, 'non_standard_modificati..."
4,Q9BQJ4,"[('SLDIWHCESTLSSDWQ', 21013, 'non_standard_mod..."


In [9]:
df_pa = pd.read_csv("all_usi_Nov2025.xlsx - all_usi.tsv", sep="\t", usecols=["Dataset", "DemodPeptide"])
df_pa.head()

,Dataset,DemodPeptide
0,PXD006633,EIVMTQSPDTLSVSPGER
1,PXD006633,EIVMTQSPDTLSVSPGER
2,PXD006633,EIVMTQSPDTLSVSPGER
3,PXD006633,EIVMTQSPDTLSVSPGER
4,PXD006633,EIVMTQSPDTLSVSPGER


In [12]:
import ast

def _parse_evidence(peptide_entry):
    if isinstance(peptide_entry, str):
        normalized = peptide_entry.replace("nan", "None")
        try:
            peptide_entry = ast.literal_eval(normalized)
        except (ValueError, SyntaxError):
            return []
    return peptide_entry

records = []
for protein, evidence in zip(df['precursor_protein'], df['evidence_peptides_scans']):
    for entry in _parse_evidence(evidence):
        peptide = entry[0]
        is_evidence = entry[3] if len(entry) > 3 else False
        status = entry[4] if len(entry) > 4 else ""
        records.append({
            'precursor_protein': protein,
            'peptide': peptide,
            'is_evidence': bool(is_evidence),
            'status': status
        })

evidence_df = pd.DataFrame(records)
missed_evidence = evidence_df[
    evidence_df['is_evidence'] &
    evidence_df['status'].astype(str).str.contains('Missed', na=False)
]

merged = missed_evidence.merge(
    df_pa,
    how='left',
    left_on='peptide',
    right_on='DemodPeptide'
)

result = merged.groupby('Dataset', dropna=False).agg(
    num_retrieved_protein=('precursor_protein', 'nunique'),
    specific_proteins=('precursor_protein', lambda vals: ';'.join(sorted(set(vals))))
).reset_index()

result.to_csv("dataset_level.tsv", sep="\t", index=False)
# result